# Inspecting the raw Pew files (.sav)

We want to combine 9 Pew Research Center surveys and use Trump approval from them as an external resource for the analysis.

The files are in SPSS format (.sav). We do not have SPSS, so we read them in Jupyter with the pyreadstat library.

## 0. Setup

In [1]:
import re
from pathlib import Path

import pandas as pd
import pyreadstat
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SURVEY_DIR = ROOT / "data" / "raw" / "survey"
print("Project root:", ROOT)
print("Survey folder:", SURVEY_DIR)

Project root: D:\могилянка\обсус\reddit_project
Survey folder: D:\могилянка\обсус\reddit_project\data\raw\survey


## 1. Find and open all .sav files

The zip archives are extracted into `data/raw/survey`.

In [2]:
SAV_NAMES = {
    "Feb17 public.sav": "Feb17",
    "Apr17 public.sav": "Apr17",
    "Typology 17 public.sav": "Typology17",
    "Jan18 public.sav": "Jan18",
    "March18 public.sav": "Mar18",
    "May18 public.sav": "May18",
    "June18 public.sav": "Jun18",
    "ATP W161.sav": "W161",
    "ATP W167.sav": "W167",
}
WAVE_ORDER = list(SAV_NAMES.values())
phone = [w for w in WAVE_ORDER if w not in ("W161", "W167")]   # 2017-2018 telephone surveys
atp = ["W161", "W167"]                                         # 2025 American Trends Panel waves

found = {}
for p in sorted(SURVEY_DIR.rglob("*.sav")):
    wave = SAV_NAMES.get(p.name)
    if wave and wave not in found:
        found[wave] = p

missing = [w for w in WAVE_ORDER if w not in found]
assert not missing, f"No .sav found for waves: {missing}. Check that the zips are extracted into {SURVEY_DIR}"

pd.DataFrame({
    "wave_id": WAVE_ORDER,
    "file (relative to data/raw/survey)": [str(found[w].relative_to(SURVEY_DIR)) for w in WAVE_ORDER],
    "size, KB": [round(found[w].stat().st_size / 1024) for w in WAVE_ORDER],
}).set_index("wave_id")

,file (relative to data/raw/survey),"size, KB"
wave_id,,
Feb17,Feb17-public\Feb17 public.sav,512
Apr17,Apr17-public-4.3-update\Apr17 public.sav,388
Typology17,Typology-17\Typology 17 public.sav,1063
Jan18,Jan18\Jan18 public.sav,390
Mar18,March18\March18 public.sav,299
May18,May18\May18 public.sav,293
Jun18,June18\June18 public.sav,419
W161,W161_Feb25\W161_Feb25\ATP W161.sav,1107
W167,W167_Apr25\W167_Apr25\ATP W167.sav,899


In [3]:
surveys, metas, labels = {}, {}, {}
for wave in WAVE_ORDER:
    df, meta = pyreadstat.read_sav(found[wave], apply_value_formats=False)
    surveys[wave] = df
    metas[wave] = meta
    labels[wave] = dict(zip(meta.column_names, meta.column_labels))  # variable name -> question text

def vlabels(wave, var):
    return metas[wave].variable_value_labels.get(var, {})

pd.DataFrame({w: {"rows (respondents)": len(d), "columns (variables)": d.shape[1]} for w, d in surveys.items()}).T

,rows (respondents),columns (variables)
Feb17,1503,127
Apr17,1501,178
Typology17,5009,163
Jan18,1503,168
Mar18,1466,108
May18,1503,123
Jun18,2002,130
W161,5086,144
W167,3589,170


The tables already have different widths, from 108 to 178 columns. These are different questionnaires.

## 2. Looking at all 9 tables

In [4]:
for w in WAVE_ORDER:
    print(f"{w}: {surveys[w].shape[0]} rows x {surveys[w].shape[1]} columns")
    display(surveys[w].head(5))

Feb17: 1503 rows x 127 columns


,psraid,sample,int_date,fcall,version,attempts,refusal,ilang,cregion,state,density,sstate,form,stimes,igender,irace,llitext0,susr,usr,scregion,qs1,q1,q1a,q2,q5af1,q5bf1,q5cf1,q5df1,q6af2,q6bf2,q6cf2,q6df2,q10a,q10b,q15af1,q15b,q15cf2,q15df1,q15ef1,q15ff1,q15gf2,q15hf2,q15if2,q16,q19,q35,q36,q37,q39,q43,q44,q45,q45vb,q45oem1,q45oem2,q45oem3,q52,q53,q54,q55,q61a,q61b,q61c,q61d,q61e,q62f1,q63f1,q64f2,q65,q66,q68f1,q69f2,q70f1,q71f2,q74,q75,q81,q82,q84a,q84bf1,q84cf1,q84df1,q84ef2,q84ff2,q84gf2,q88,q90f1,q91f2,sex,age,gen5,educ2,hisp,adults,racethn,racethn2,birth_hisp,citizen,child,relig,chr,born,attend,q92,q92a,income,reg,party,partyln,partysum,partyideo,q93,q94,ideo,hh1,hh3,ql1,ql1a,qc1,money2,money3,iphoneuse,hphoneuse,ll,cp,cellweight,weight
0,100008.0,1.0,21017.0,170207.0,2.0,4.0,0.0,1.0,2.0,17.0,4.0,17.0,2.0,4.0,2.0,1.0,1.0,S,S,2.0,NaN,2.0,1.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,2.0,1.0,NaN,2.0,2.0,NaN,NaN,NaN,2.0,2.0,1.0,3.0,4.0,1.0,2.0,3.0,1.0,1.0,1.0,NaN,,NaN,NaN,NaN,2.0,2.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,2.0,1.0,2.0,NaN,1.0,NaN,2.0,1.0,NaN,1.0,3.0,4.0,NaN,NaN,NaN,2.0,4.0,2.0,4.0,NaN,1.0,2.0,80.0,2.0,6.0,2.0,2.0,1.0,1.0,NaN,NaN,0.0,1.0,NaN,2.0,2.0,2.0,NaN,9.0,1.0,3.0,2.0,2.0,3.0,NaN,1.0,4.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,NaN,1.733333
1,100019.0,1.0,21217.0,170207.0,2.0,4.0,1.0,1.0,3.0,37.0,4.0,37.0,1.0,4.0,1.0,3.0,1.0,U,U,3.0,NaN,2.0,1.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,2.0,2.0,NaN,1.0,2.0,2.0,NaN,NaN,NaN,4.0,4.0,1.0,2.0,4.0,3.0,2.0,2.0,1.0,I DON'T THINK HE'S THE BEST MAN FOR THE JOB,11.0,NaN,NaN,2.0,2.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,3.0,3.0,NaN,1.0,2.0,2.0,NaN,1.0,NaN,1.0,NaN,1.0,1.0,4.0,4.0,4.0,2.0,NaN,NaN,NaN,3.0,3.0,NaN,2.0,70.0,3.0,8.0,2.0,2.0,1.0,1.0,NaN,NaN,0.0,5.0,NaN,NaN,3.0,2.0,NaN,7.0,1.0,2.0,NaN,2.0,4.0,NaN,2.0,3.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,NaN,1.500000
2,100020.0,1.0,21217.0,170207.0,2.0,4.0,1.0,1.0,1.0,36.0,4.0,36.0,1.0,4.0,2.0,3.0,2.0,S,S,1.0,NaN,2.0,1.0,2.0,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,2.0,2.0,NaN,2.0,1.0,2.0,NaN,NaN,NaN,3.0,3.0,1.0,2.0,4.0,3.0,1.0,2.0,1.0,I THINK HE IS TOO MUCH TIES WITH LARGE CORPORATIONS,41.0,NaN,NaN,2.0,2.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,2.0,3.0,NaN,1.0,2.0,2.0,NaN,1.0,NaN,1.0,NaN,1.0,1.0,4.0,2.0,4.0,2.0,NaN,NaN,NaN,3.0,2.0,NaN,2.0,69.0,3.0,8.0,2.0,2.0,1.0,1.0,NaN,NaN,0.0,2.0,NaN,2.0,2.0,1.0,3.0,8.0,1.0,3.0,2.0,2.0,3.0,NaN,2.0,4.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,NaN,1.533333
3,100021.0,1.0,20717.0,170207.0,1.0,1.0,0.0,1.0,2.0,27.0,5.0,27.0,2.0,1.0,2.0,4.0,1.0,U,U,2.0,NaN,1.0,2.0,1.0,NaN,NaN,NaN,NaN,1.0,1.0,2.0,2.0,1.0,2.0,NaN,1.0,1.0,NaN,NaN,NaN,1.0,1.0,1.0,3.0,2.0,1.0,1.0,3.0,2.0,2.0,1.0,NaN,,NaN,NaN,NaN,1.0,2.0,2.0,2.0,1.0,1.0,1.0,2.0,2.0,NaN,NaN,1.0,3.0,2.0,NaN,1.0,NaN,2.0,2.0,1.0,2.0,1.0,2.0,NaN,NaN,NaN,2.0,2.0,2.0,3.0,NaN,NaN,1.0,50.0,4.0,3.0,2.0,2.0,1.0,1.0,NaN,NaN,0.0,2.0,NaN,2.0,4.0,2.0,NaN,7.0,1.0,1.0,NaN,1.0,1.0,2.0,NaN,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,NaN,5.866667
4,100024.0,1.0,20717.0,170207.0,1.0,1.0,0.0,1.0,2.0,17.0,5.0,17.0,1.0,1.0,1.0,1.0,1.0,S,S,2.0,NaN,2.0,1.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,2.0,1.0,2.0,2.0,NaN,2.0,2.0,2.0,NaN,NaN,NaN,4.0,4.0,2.0,2.0,4.0,1.0,3.0,2.0,1.0,WE DON'T KNOW ANYTHING ABOUT HIM,21.0,NaN,NaN,2.0,2.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,NaN,1.0,2.0,1.0,NaN,1.0,NaN,1.0,NaN,2.0,2.0,4.0,3.0,3.0,1.0,NaN,NaN,NaN,4.0,1.0,NaN,2.0,70.0,3.0,6.0,2.0,2.0,2.0,2.0,NaN,NaN,0.0,1.0,NaN,1.0,1.0,1.0,3.0,7.0,1.0,2.0,NaN,2.0,5.0,NaN,2.0,4.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,NaN,1.700000


Apr17: 1501 rows x 178 columns


,psraid,sample,int_date,fcall,attempts,refusal,ilang,version,cregion,state,density,usr,scregion,sstate,susr,stimes,igender,irace,form,llitext0,qs1,q1,q1a,q2,q3,q11f1,q12,q13,q19f2,q20f1,q21bf2,q21df2,q25af1,q25bf1,q25cf2,q25df2,q25ef2,q26,q30f2,q31f1,q32f2m1,q32f2m2,q32f2m3,q32f2m4,q32f2m5,q32f2m6,q32f2m7,q32f2m8,q32f2m9,q32f2m10,q32f2m11,q32f2m12,q32f2oe1,q32f2oe2,q32f2oe3,q32f2oe4,q32f2oe5,q32f2oe6,q32f2oe7,q32f2oe8,q32f2oe9,q32f2oe10,q32f2oe11,q32f2oe12,q36a,q36b,q36c,q37a,q37b,q41af1,q41bf1,q41cf1,q41df1,q41ef1,q41ff1,q41gf1,q41hf2,q41if2,q41jf2,q41kf2,q41lf2,q41mf2,q41nf2,q42f1,q43f2,q46f2,q47f1,q48,q49,q50f1,q50f1oe_1,q50f1oe_2,q55f2,q56a,q56b,q56c,q56d,q57a,q57b,q57c,q57d,q61af1,q61bf1,q61cf1,q61df1,q61ef1,q61ff1,q61gf2,q61hf2,q61if2,q61jf2,q61kf2,q61lf2,q62f1,q63f2,q65f2,q66f1,q67a,q67b,q67c,q67d,q67e,q70,q71,q72,q80,q81,q82,q83,q84,q92a,q92b,q92c,q92ff1,q92gf1,q92hf2,q92if2,q95,q96,sex,age,gen5,educ2,hisp,citizen,race3m1,race3m2,race3m3,racethn,racethn2,birth_hisp,relig,chr,born,attend,income,reg,party,partyln,partysum,q98,ideo,partyideo,hh1,hh3,adults,child,ql1,ql1a,qc1,ll,cp,money2,ckinfo,iphoneuse,hphoneuse,utweight,weight
0,100005.0,1.0,40517.0,170405.0,1.0,0.0,1.0,1.0,2.0,29.0,4.0,S,2.0,29.0,S,1.0,1.0,1.0,2.0,1.0,NaN,2.0,1.0,2.0,2.0,NaN,2.0,3.0,NaN,NaN,2.0,1.0,NaN,NaN,4.0,3.0,3.0,3.0,2.0,NaN,97.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,3.0,3.0,3.0,3.0,3.0,NaN,9.0,2.0,NaN,2.0,2.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,3.0,2.0,2.0,3.0,NaN,9.0,2.0,NaN,1.0,1.0,1.0,1.0,1.0,NaN,9.0,9.0,1.0,9.0,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,1.0,1.0,3.0,NaN,2.0,72.0,2.0,6.0,2.0,NaN,2.0,NaN,NaN,2.0,2.0,NaN,1.0,NaN,2.0,2.0,5.0,1.0,3.0,2.0,2.0,NaN,4.0,3.0,2.0,2.0,2.0,0.0,2.0,1.0,NaN,1.0,0.0,NaN,NaN,1.0,2.0,1.00,2.941176
1,100010.0,1.0,40517.0,170405.0,1.0,0.0,1.0,1.0,3.0,48.0,1.0,R,3.0,48.0,R,1.0,1.0,4.0,2.0,2.0,NaN,1.0,2.0,1.0,2.0,NaN,1.0,3.0,NaN,NaN,1.0,2.0,NaN,NaN,2.0,1.0,1.0,1.0,1.0,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,3.0,1.0,3.0,2.0,3.0,NaN,2.0,1.0,NaN,2.0,2.0,NaN,NaN,NaN,1.0,2.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,3.0,1.0,1.0,1.0,NaN,2.0,3.0,NaN,1.0,1.0,1.0,1.0,4.0,NaN,2.0,4.0,2.0,2.0,NaN,NaN,NaN,2.0,2.0,2.0,NaN,NaN,2.0,2.0,2.0,2.0,2.0,59.0,3.0,5.0,1.0,NaN,1.0,NaN,NaN,3.0,3.0,1.0,1.0,NaN,1.0,1.0,2.0,1.0,1.0,NaN,1.0,1.0,2.0,1.0,1.0,NaN,1.0,0.0,1.0,NaN,NaN,1.0,1.0,NaN,NaN,2.0,2.0,0.45,1.323529
2,100021.0,1.0,40517.0,170405.0,1.0,0.0,1.0,1.0,3.0,12.0,4.0,S,3.0,12.0,S,1.0,2.0,2.0,1.0,1.0,NaN,2.0,1.0,2.0,2.0,2.0,2.0,3.0,NaN,2.0,NaN,NaN,4.0,4.0,NaN,NaN,NaN,3.0,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,2.0,3.0,1.0,NaN,9.0,1.0,1.0,3.0,3.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,1.0,1.0,2.0,NaN,NaN,NaN,NaN,1.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,1.0,1.0,1.0,1.0,4.0,1.0,NaN,9.0,9.0,2.0,3.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,2.0,NaN,NaN,2.0,8.0,1.0,82.0,2.0,8.0,2.0,NaN,1.0,NaN,NaN,1.0,1.0,NaN,13.0,NaN,2.0,5.0,10.0,1.0,2.0,NaN,2.0,NaN,4.0,5.0,2.0,2.0,2.0,0.0,9.0,9.0,NaN,1.0,1.0,NaN,NaN,2.0,2.0,0.42,1.235294
3,100028.0,1.0,40517.0,170405.0,1.0,0.0,1.0,1.0,3.0,24.0,3.0,S,3.0,24.0,S,1.0,1.0,2.0,2.0,2.0,NaN,1.0,1.0,1.0,2.0,NaN,3.0,4.0,NaN,NaN,1.0,2.0,NaN,NaN,1.0,1.0,2.0,1.0,1.0,NaN,3.0,4.0,98.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,2.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,2.0,3.0,2.0,2.0,2.0,NaN,2.0,1.0,NaN,2.0,1.0,NaN,NaN,NaN,1.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,NaN,2.0,4.0,NaN,2.0,1.0,4.0,2.0,3.0,NaN,2.0,3.0,1.0,3.0,NaN,NaN,NaN,1.0,2.0,1.0,NaN,NaN,1.0,2.0,2.0,8.0,2.0,62.0,3.0,5.0,2.0,NaN,1.0,NaN,NaN,1.0,1.0,NaN,1.0,NaN,2.0,4.0,6.0,1.0

Typology17: 5009 rows x 163 columns


...

,respid,sample,int_date,fcall,attempt,refusal,ilang,cregion,state,density,usr,scregion,sstate,susr,sdensity,timezone,igender,ihisp1,irace1m1,irace1m2,irace1m3,irace1m4,form,llitext,phtype,qs1,q1,q2,q6f2,q6f2_oe1,q6f2_oe2,q6f2_oe3,q7a,q7b,q7c,q7df1,q7ef2,q10a,q10b,q11,q12,campnii,q20,q21,cheat,q25f1,Q25F1_1,Q25F1_2,Q25F1_3,q26,q27,q28,q29,q30,q31,q32,q38,q39,q40,q41a,q41bf1,q41cf1,q41df1,q41ef1,q41ff1,q41gf2,q41hf2,q41if2,q41jf2,q41lf2,q42,q60af1,q60bf1,q60cf1,q60df1,q60ef1,q60ff1,q60gf1,q60jf2,q60kf2,q60lf2,q60mf2,q60nf2,q60of2,q68,q69,q82,q83,q90a,q90c,q91,q92,q93,q94,q100,q101,q102,q103,q104,sex,age,educ,hisp,racecmb,racethn,birth_hisp,relig,chr,born,attend,income,reg,party,partyln,q105,partysum,ideo,partyideo,partysumideo,hh1,hh3,adults,ql1,ql1a,qc1,ll,cp,money2,cellweight,weight
0,1.0,1.0,180605.0,180605.0,1.0,0.0,1.0,3.0,11.0,5.0,U,3.0,11.0,U,5.0,1.0,2.0,2.0,2.0,NaN,NaN,NaN,2.0,1.0,0,NaN,2.0,2.0,1.0,47.0,95.0,43.0,3.0,1.0,1.0,NaN,1.0,2.0,1.0,10.0,2.0,1.0,2.0,NaN,2.0,NaN,NaN,NaN,NaN,9.0,2.0,2.0,2.0,3.0,2.0,4.0,1.0,3.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN,2.0,4.0,2.0,4.0,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,2.0,2.0,9.0,2.0,9.0,1.0,1.0,2.0,1.0,1.0,2.0,3.0,2.0,9.0,1.0,2.0,1.0,1.0,4.0,1.0,78.0,6.0,2.0,2.0,2.0,NaN,1.0,NaN,2.0,2.0,9.0,1.0,2.0,NaN,NaN,2.0,3.0,4.0,4.0,2.0,2.0,2.0,1.0,NaN,NaN,1.0,1.0,NaN,NaN,0.581332
1,2.0,1.0,180605.0,180605.0,1.0,0.0,1.0,3.0,10.0,4.0,S,3.0,10.0,S,4.0,1.0,2.0,2.0,1.0,NaN,NaN,NaN,1.0,1.0,1,NaN,2.0,9.0,NaN,NaN,NaN,NaN,2.0,2.0,2.0,2.0,NaN,1.0,9.0,12.0,13.0,3.0,1.0,NaN,1.0,9.0,99.0,NaN,NaN,9.0,2.0,2.0,9.0,2.0,1.0,NaN,2.0,3.0,4.0,9.0,2.0,9.0,9.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,2.0,4.0,9.0,9.0,9.0,4.0,9.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,9.0,1.0,2.0,2.0,2.0,2.0,2.0,3.0,2.0,2.0,2.0,9.0,2.0,1.0,78.0,4.0,2.0,1.0,1.0,NaN,1.0,NaN,2.0,4.0,2.0,1.0,3.0,1.0,3.0,1.0,2.0,3.0,1.0,2.0,2.0,2.0,2.0,1.0,NaN,1.0,0.0,NaN,NaN,1.559586
2,4.0,1.0,180605.0,180605.0,1.0,0.0,1.0,3.0,12.0,2.0,S,3.0,12.0,S,2.0,1.0,2.0,2.0,2.0,NaN,NaN,NaN,1.0,2.0,0,NaN,1.0,2.0,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,NaN,2.0,2.0,2.0,3.0,1.0,1.0,NaN,1.0,1.0,31.0,37.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,4.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,3.0,3.0,1.0,2.0,3.0,3.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0,2.0,63.0,3.0,2.0,1.0,1.0,NaN,13.0,NaN,1.0,3.0,1.0,1.0,1.0,NaN,3.0,1.0,1.0,1.0,1.0,3.0,3.0,3.0,2.0,1.0,NaN,1.0,0.0,NaN,NaN,2.428216
3,6.0,1.0,180605.0,180605.0,1.0,0.0,1.0,1.0,23.0,1.0,S,1.0,23.0,S,1.0,1.0,1.0,2.0,3.0,NaN,NaN,NaN,1.0,2.0,1,NaN,1.0,1.0,NaN,NaN,NaN,NaN,4.0,1.0,4.0,1.0,NaN,1.0,2.0,7.0,1.0,3.0,1.0,NaN,1.0,1.0,33.0,NaN,NaN,2.0,1.0,2.0,2.0,1.0,9.0,NaN,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,2.0,1.0,2.0,1.0,2.0,4.0,2.0,1.0,2.0,1.0,1.0,4.0,2.0,1.0,56.0,2.0,2.0,5.0,4.0,NaN,2.0,NaN,2.0,5.0,4.0,1.0,3.0,1.0,1.0,1.0,2.0,3.0,1.0,2.0,2.0,2.0,1.0,NaN,NaN,1.0,1.0,NaN,NaN,1.026937
4,7.0,1.0,180605.0,180605.0,1.0,0.0,1.0,1.0,36.0,2.0,R,1.0,36.0,R,2.0,1.0,1.0,1.0,2.0,NaN,NaN,NaN,1.0,1.0,0,NaN,2.0,1.0,NaN,NaN,NaN,NaN,9.0,1.0,1.0,1.0,NaN,1.0,2.0,6.0,1.0,4.0,1.0,NaN,NaN,1.0,2.0,68.0,NaN,1.0,1.0,1.0,1.0,1.0,2.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,1.0,1.0,2.0,1.0,9.0,9.0,2.0,3.0,3.0,NaN,9.0,9.0,4.0,1.0,99.0,9.0,9.0,9.0,9.0,NaN,99.0,9.0,NaN,9.0,10.0,9.0,9.0,9.0,NaN,9.0,9.0,9.0,9.0,9.0,9.0,2.0,9.0,9.0,NaN,1.0,1.0,NaN,NaN,0.371928


W161: 5086 rows x 144 columns


,QKEY,INTERVIEW_START_W161,INTERVIEW_END_W161,DEVICE_TYPE_W161,SVYMODE_W161,LANG_W161,FORM_W161,X_PARTYSUMFINAL_W161,SATIS_W161,YEARAHEAD_W161,POL1DT_W161,POL1DTSTR_W161,PRESWHERE_W161,DTSUCCESS_W161,QUALPRES_TRMP_MF_W161,QUALPRES_TRMP_PHY_W161,QUALPRES_TRMP_ADV_W161,QUALPRES_TRMP_ETH_W161,QUALPRES_TRMP_RSP_W161,QUALPRES_TRMP_FIRM_W161,PRES_WASH_W161,PRES_WASH_EXP_W161,PRESCAB_W161,VP2PRES_W161,VP_INFL_W161,ETHICSDJT_W161,NATPROBS_GUN_W161,NATPROBS_INFR_W161,NATPROBS_IMM_W161,NATPROBS_HC_W161,NATPROBS_MNPOL_W161,NATPROBS_AI_W161,NATPROBS_CLIM_W161,NATPROBS_DEF_W161,NATPROBS_VCRI_W161,NATPROBS_K12_W161,NATPROBS_POVTY_W161,NATPROBS_LONE_W161,NATPROBS_INFL_W161,NATPROBS_ITERR_W161,NATPROBS_COMP_W161,NATPROBS_MOR_W161,NATPROBS_SSMED_W161,NATPROBS_NTDIS_W161,NATPROBS_RAC_W161,NATPROBS_UNEM_W161,NATPROBS_DTERR_W161,NATPROBS_DRG_W161,NATPROBS_PLSYS_W161,NATPROBS_FCT_W161,NEWPRES_ACCOMP_W161,NEWPRES_SUPP_W161,NEWPRES_ACTNS_W161,PRESWITHOP_W161,OPWITHPRES_W161,ECON1_W161,ECON1B_W161,ECON_FUT_JBS_W161,ECON_FUT_PRICE_W161,ECON_FUT_ENRG_W161,ECON_FUT_HOUS_W161,ECON_FUT_HLTH_W161,FAVPOL_TRUMP_W161,FAVPOL_VANCE_W161,FAVPOL_MJOHNSON_W161,FAVPOL_JEFFRIES_W161,FAVPOL_ZUCKERBERG_W161,FAVPOL_THUNE_W161,FAVPOL_SCHUMER_W161,FAVPOL_MUSK_W161,FAVPOL_RFKJR_W161,EXCPWER_W161,EXCPWER_NAME_W161,MARIJUANA3_W161,TAXRATES400_W161,TAXRATESBUS_W161,CONF2_CAREER_W161,CONF2_APPNT_W161,FEDSTATE_STAT_W161,FEDSTATE_RGHT_W161,FEDSTATE_FED2_W161,CLRSOL_W161,PRESGRPS_BP_W161,PRESGRPS_HP_W161,PRESGRPS_WP_W161,PRESGRPS_AAP_W161,PRESGRPS_EC_W161,PRESGRPS_MEN_W161,PRESGRPS_WMN_W161,PRESGRPS_UN_W161,PRESGRPS_LGB_W161,PRESGRPS_BC_W161,PRESGRPS_MIL_W161,PRESGRPS_OLD_W161,PRESGRPS_YNG_W161,PRESGRPS_PR_W161,PRESGRPS_CHI_W161,PRESGRPS_WLTH_W161,PRESGRPS_TRANS_W161,PRESGRPS_YOU_W161,REPWORKTRMP_W161,EOFOL_W161,BIRTHCIT_APP_W161,IMMIG_DT_APP_MILBRD_W161,IMMIG_DT_APP_ASYLUM_W161,IMMIG_DT_APP_DEPORT_W161,IMMIG_DT_APP_CUTFND_W161,DEPRT_DEG_W161,JAN6PARDAPP_NONVIOL_W161,JAN6PARDAPP_VIOL_W161,JAN6PARDAPP_JB_W161,JAN6CONFCM_MOD_W161,F_METRO,F_CREGION,F_CDIVISION,F_USR_SELFID,F_AGECAT,F_GENDER,F_EDUCCAT,F_EDUCCAT2,F_HISP,F_HISP_ORIGIN,F_YEARSINUS_RECODE,F_RACECMB,F_RACETHNMOD,F_BIRTHPLACE,F_MARITAL,F_RELIG,F_BORN,F_RELIGCAT1,F_ATTENDPER,F_PARTY_FINAL,F_PARTYLN_FINAL,F_PARTYSTR_FINAL,F_PARTYLNCLOSE_FINAL,F_PARTYSUM_FINAL,F_PARTYSUMIDEO_FINAL,F_REG,F_INC_SDT1,F_IDEO,F_INTFREQ,F_VOLSUM,F_INC_TIER2,WEIGHT_W161
0,100803.0,28-Jan-2025 21:38:38,28-Jan-2025 21:48:06,3.0,1.0,9.0,1.0,1.0,2.0,NaN,1.0,1.0,1.0,NaN,2.0,2.0,2.0,2.0,2.0,2.0,1.0,NaN,1.0,NaN,NaN,1.0,1.0,3.0,1.0,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,3.0,2.0,NaN,3.0,1.0,2.0,1.0,2.0,2.0,NaN,2.0,1.0,99.0,99.0,99.0,NaN,NaN,NaN,NaN,1.0,NaN,3.0,3.0,5.0,2.0,1.0,2.0,3.0,3.0,1.0,3.0,2.0,3.0,3.0,NaN,NaN,NaN,NaN,2.0,1.0,1.0,1.0,3.0,NaN,NaN,NaN,NaN,3.0,1.0,4.0,3.0,2.0,3.0,4.0,3.0,1.0,1.0,3.0,3.0,4.0,2.0,2.0,4.0,3.0,4.0,2.0,1.0,5.0,2.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,NaN,1.0,NaN,1.0,1.0,1.0,9.0,2.0,4.0,1.0,3.0,1.046094
1,103379.0,29-Jan-2025 12:54:51,29-Jan-2025 13:14:08,3.0,1.0,9.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,2.0,2.0,2.0,2.0,2.0,1.0,1.0,NaN,1.0,NaN,NaN,1.0,3.0,2.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,1.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,3.0,2.0,NaN,2.0,3.0,3.0,4.0,3.0,2.0,NaN,1.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,1.0,NaN,3.0,5.0,5.0,3.0,2.0,5.0,4.0,3.0,1.0,3.0,2.0,3.0,3.0,NaN,NaN,NaN,NaN,2.0,3.0,1.0,1.0,3.0,NaN,NaN,NaN,NaN,3.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0,3.0,1.0,4.0,4.0,2.0,1.0,3.0,5.0,2.0,4.0,1.0,2.0,3.0,2.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,NaN,1.0,NaN,1.0,1.0,1.0,9.0,2.0,5.0,2.0,3.0,0.992757
2,103538.0,28-Jan-2025 16:24:57,28-Jan-2025 16:42:25,3.0,1.0,9.0,2.0,2.0,NaN,2.0,2.0,1.0,NaN,2.0,5.0,4.0,5.0,5.0,5.0,5.0,NaN,2.0,NaN,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,1.0,1.0,99.0,1.0,3.0,4.0,4.0,NaN,2.0,2.0,2.0,3.0,4.0,3.0,NaN,3.0,4.0,NaN,NaN,NaN,NaN,3.0,2.0,4.0,4.0,NaN,2.0,1.0,1.0,1.0,1.0,3.0,4.0,3.0,

W167: 3589 rows x 170 columns


,QKEY,INTERVIEW_START_W167,INTERVIEW_END_W167,DEVICE_TYPE_W167,SVYMODE_W167,LANG_W167,FORM_W167,POL1DT_W167,POL1DTSTR_W167,LIFEFIFTY_W167,USCONF_FUT_W167,WISDOM_PPL_W167,GOVSIZE1_W167,DTCONF_ECON_W167,DTCONF_IMMI_W167,DTCONF_TRAD_W167,DTCONF_ABCN_W167,DTCONF_CLSR_W167,DTCONF_HECR_W167,DTCONF_FRGN_W167,DTCONF_HCP_W167,DTCONF_TAX_W167,DTCONF_EXEC_W167,NEW_ADMIN_ECON_W167,NEW_ADMIN_GLOB_W167,DTTRUST_W167,PRES_ETHICAL_ADMIN_W167,DEMACT_IMPD_CMN_W167,DEMACT_IMPD_PUSH_W167,DEMOFF_EFF_CMN_W167,DEMOFF_EFF_PUSH_W167,TRUMPACT_IMPD_CMN_W167,TRUMPACT_IMPD_PUSH_W167,TRUMP_EFF_CMN_W167,TRUMP_EFF_PUSH_W167,DT_REPRE_W167,ECON1_W167,ECON1B_W167,PERSFNC_W167,PERSFNCB_W167,ECONCONC_UNEM_W167,ECONCONC_PRICE_W167,ECONCONC_REAL_W167,ECONCONC_STCK_W167,ECONCONC_ENG2_W167,POLINTOL2_REP_W167,POLINTOL2_DEM_W167,GOVRESP_COL_W167,GOVRESP_INS_W167,GOVRESP_STAND_W167,GOVRESP_RET_W167,GOVRESP_EDUC_W167,GOVRESP_WIFI_W167,GOVRESP_CLEA_W167,GOVRESP_ENERG_W167,GOVRESP2_MIL_W167,GOVRESP2_BORD_W167,GOVRESP2_PARK_W167,INSTFAV_CONG_W167,INSTFAV_COURT_W167,INSTFAV_REP_W167,INSTFAV_DEM_W167,FEDCRT_IMPT_W167,FEDCRT_CONF_W167,FEDCRT_CONF_NS_W167,TARIFF_APP_W167,TRUMPAPPROV_DEI_W167,TRUMPAPPROV_DEI_NS_W167,POLSTOR_FOLL_LAW_W167,POLSTOR_FOLL_DOGE_W167,POLSTOR_FOLL_DPRT_W167,POLSTOR_FOLL_TARF_W167,POLSTOR_FOLL_PROT_W167,POLSTOR_FOLL_MEAS_W167,POLSTOR_FOLL_CUTS_W167,POLSTOR_FOLL_BORD_W167,POLSTOR_FOLL_LEGAL_W167,POLSTOR_FOLL_SEC_W167,DT_REDUCTIONS_W167,DT_RED_CARE_W167,DT_RED_RUN_W167,DT_RED_NEED_W167,DT_RED_SAV_W167,DTEOS_PRESPOW2_W167,DISCRRIM_BLACK_W167,DISCRRIM_HISP_W167,DISCRRIM_WHITE_W167,DISCRRIM_ASIAN_W167,DISCRRIM_WOM_W167,DISCRRIM_MEN_W167,DISCRRIM_TRANS_W167,DISCRRIM_IMMIGL_W167,DISCRRIM_IMMIGU_W167,DISCRRIM_NOCOLL_W167,DISCRRIM_LGBC_W167,DISCRRIM_REL_W167,DISCRRIM_RUR_W167,DISCRRIM_CIT_W167,DISCRRIM_YOUNG_W167,DISCRRIM_OLD_W167,DISCRRIM_JEW_W167,DISCRRIM_MUS_W167,DISCRRIM_EVANG_W167,DISCRRIM_ATH_W167,PF_HAPPEN_HOU_W167,PF_HAPPEN_LOSE_W167,PF_HAPPEN_RAISE_W167,PF_HAPPEN_SAVE_W167,PF_HAPPEN_MED_W167,PF_HAPPEN_FOOD_W167,PF_HAPPEN_CHILD_W167,PF_HAPPEN_AUTO_W167,PF_HAPPEN_BRRW_W167,PF_HAPPEN_VACA_W167,PF_HAPPEN_PYDY_W167,BILLSTYPICAL_W167,BILLSNOW_W167,EMERGFUND_W167,FEDCRT_PROB_W167,FEDCRT_FOL_W167,SCOTUS_RULFOL_W167,TRLYAM_TGEN_W167,TRLYAM_SGENLG_W167,TRLYAM_SGENIL_W167,IMM_ENG_RALLY_W167,IMM_ENG_ACTION_W167,IMM_ENG_POST_W167,IMM_ENG2_RALLY_W167,IMM_ENG2_ACTION_W167,IMM_ENG2_POST_W167,IMM_APP_W167,HORSE_W153,VOTESTRONG_W153,VOTEGEN24_LEAN_W153,HORSE_W156,VOTESTRONG_W156,VOTEGEN24_LEAN_W156,VOTEGEN_POST_W159,F_METRO,F_CREGION,F_CDIVISION,F_USR_SELFID,F_AGECAT,F_GENDER,F_EDUCCAT,F_EDUCCAT2,F_HISP,F_HISP_ORIGIN,F_YEARSINUS_RECODE,F_RACECMB,F_RACETHNMOD,F_BIRTHPLACE,F_MARITAL,F_RELIG,F_BORN,F_RELIGCAT1,F_ATTENDPER,F_PARTY_FINAL,F_PARTYLN_FINAL,F_PARTYSTR_FINAL,F_PARTYLNCLOSE_FINAL,F_PARTYSUM_FINAL,F_PARTYSUMIDEO_FINAL,F_REG,F_INC_SDT1,F_IDEO,F_INTFREQ,F_VOLSUM,F_INC_TIER2,WEIGHT_W167
0,100363.0,08-Apr-2025 15:12:07,08-Apr-2025 15:24:20,1.0,1.0,9.0,2.0,2.0,1.0,2.0,NaN,3.0,1.0,4.0,4.0,NaN,NaN,NaN,NaN,4.0,4.0,4.0,4.0,2.0,2.0,2.0,NaN,1.0,1.0,3.0,3.0,1.0,4.0,4.0,1.0,1.0,3.0,2.0,3.0,2.0,3.0,NaN,NaN,1.0,1.0,1.0,2.0,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,3.0,3.0,3.0,1.0,NaN,4.0,4.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,3.0,3.0,3.0,4.0,3.0,4.0,NaN,NaN,2.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,3.0,2.0,2.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,4.0,4.0,2.0,2.0,2.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,1.0,NaN,1.0,1.0,1.0,2.0,2.0,2.0,NaN,NaN,NaN,5.0,2.0,2.0,NaN,2.0,2.0,NaN,2.0,1.0,1.0,2.0,2.0,2.0,2.0,1.0,6.0,2.0,NaN,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,4.0,2.0,NaN,2.0,NaN,2.0,3.0,1.0,9.0,3.0,2.0,1.0,2.0,1.135669
1,103379.0,08-Apr-2025 14:50:19,08-Apr-2025 15:44:18,3.0,1.0,9.0,2.0,1.0,1.0,3.0,NaN,3.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,2.0,2.0,1.0,1.0,1.0,NaN,4.0,4.0,4.0,3.0,3.0,1.0,2.0,1.0,1.0,2.0,3.0,2.0,3.0,3.0,NaN,NaN,4.0,3.0,2.0,1.0,NaN,NaN,NaN,NaN,2.0,2.0,1.0,2.0,1.0,1.0,2.0,3.0,2.0,2.0,4.0,1.0,NaN,4.0,2.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,5.0,4.0,3.0,3.0,5.0,1.0,NaN,NaN,1.0,1.0,3.0,Na

the phone files (2017-2018) start with technical columns like `respid`, `sample`, `int_date`; the ATP files (2025) start with `QKEY` and names ending in `_W161` / `_W167`. Everything is stored as numbers.

The table holds only a number, what it means is stored in the metadata. Example: the approval question in Apr17 (`q1`):

In [5]:
print(labels["Apr17"]["q1"])
print(vlabels("Apr17", "q1"))
surveys["Apr17"]["q1"].head(8).tolist()

Q.1 Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK. IF DEPENDS PROBE ONCE WITH:  Overall do you approve or disapprove of the way Donald Trump is handling his job as President?
{1.0: 'Approve', 2.0: 'Disapprove', 9.0: "Don't know/Refused (VOL.)"}


[2.0, 1.0, 2.0, 1.0, 9.0, 2.0, 2.0, 1.0]

## 3. The columns differ between files

In [6]:
pd.DataFrame({w: list(surveys[w].columns[:8]) for w in WAVE_ORDER})

,Feb17,Apr17,Typology17,Jan18,Mar18,May18,Jun18,W161,W167
0,psraid,psraid,mergeid,respid,masterid,respid,respid,QKEY,QKEY
1,sample,sample,sample,sample,samptype,sample,sample,INTERVIEW_START_W161,INTERVIEW_START_W167
2,int_date,int_date,int_date,int_date,ftcalldt,int_date,int_date,INTERVIEW_END_W161,INTERVIEW_END_W167
3,fcall,fcall,phase,fcall,numatts,fcall,fcall,DEVICE_TYPE_W161,DEVICE_TYPE_W167
4,version,attempts,fcall,attempt,refcon,attempt,attempt,SVYMODE_W161,SVYMODE_W167
5,attempts,refusal,attempts,refusal,ilang,refusal,refusal,LANG_W161,LANG_W167
6,refusal,ilang,refusal,ilang,cregion,ilang,ilang,FORM_W161,FORM_W167
7,ilang,version,stimes,cregion,STATE,cregion,cregion,X_PARTYSUMFINAL_W161,POL1DT_W167


In [7]:
colsets = {w: {c.lower() for c in d.columns} for w, d in surveys.items()}

common_all = set.intersection(*colsets.values())
common_phone = set.intersection(*[colsets[w] for w in phone])
union_all = set.union(*colsets.values())

print(f"Unique column names in total: {len(union_all)}")
print(f"Shared by all 9 files: {len(common_all)}")
print(f"Shared by the 7 phone files: {len(common_phone)}")
print(f"Shared between phone and ATP files: {len(common_phone & colsets['W161'])}")
print()
print("The 30 names shared by the phone files:")
print(sorted(common_phone))

Unique column names in total: 888
Shared by all 9 files: 0
Shared by the 7 phone files: 30
Shared between phone and ATP files: 0

The 30 names shared by the phone files:
['age', 'attend', 'birth_hisp', 'born', 'chr', 'cp', 'cregion', 'density', 'hh1', 'hh3', 'hisp', 'ideo', 'igender', 'ilang', 'income', 'll', 'party', 'partyideo', 'partyln', 'partysum', 'racethn', 'reg', 'relig', 'scregion', 'sex', 'sstate', 'state', 'susr', 'usr', 'weight']


Phone surveys and ATP waves share **no column names at all**, and even among the phone files only 30 columns match. This is why we need a crosswalk

### The full codebook: every variable of every file

In [8]:
rows = []
for w in WAVE_ORDER:
    d = surveys[w]
    for c in d.columns:
        rows.append({"wave": w, "variable": c, "question text": labels[w].get(c) or "",
                     "unique values": d[c].nunique(), "% missing": round(100 * d[c].isna().mean(), 1),
                     "labelled codes": len(vlabels(w, c))})
codebook = pd.DataFrame(rows)
print("Variables in total (all 9 files):", len(codebook))
codebook.groupby("wave", sort=False).size().to_frame("variables").T

Variables in total (all 9 files): 1311


wave,Feb17,Apr17,Typology17,Jan18,Mar18,May18,Jun18,W161,W167
variables,127,178,163,168,108,123,130,144,170


wave,Feb17,Apr17,Typology17,Jan18,Mar18,May18,Jun18,W161,W167
variables,127,178,163,168,108,123,130,144,170


In [9]:
pd.set_option("display.max_rows", None)
for w in WAVE_ORDER:
    print(f"\n\n\n\n================== {w}: all {surveys[w].shape[1]} variables")
    display(codebook[codebook["wave"] == w].drop(columns="wave").set_index("variable"))





================== Feb17: all 127 variables


,question text,unique values,% missing,labelled codes
variable,,,,
psraid,Unique ID,1503,0.0,0
sample,Sample type,2,0.0,4
int_date,Interview date,6,0.0,0
fcall,Date of first call to number,6,0.0,0
version,Revisions to questionnaire,2,0.0,2
attempts,Total attempts to number,8,0.0,0
refusal,Refusal Conversion,2,0.0,2
ilang,Language of interview,2,0.0,2
cregion,Final Census Region after zip/fips merges,4,0.0,4






================== Apr17: all 178 variables


,question text,unique values,% missing,labelled codes
variable,,,,
psraid,Unique ID,1501,0.0,0
sample,Sample type,2,0.0,4
int_date,Interview date,7,0.0,0
fcall,Date of first call to number,5,0.0,0
attempts,Total attempts to number,9,0.0,0
refusal,Refusal Conversion,2,0.0,2
ilang,Language of interview,2,0.0,2
version,Revisions to questionnaire,3,0.0,3
cregion,Final Census Region after zip/fips merges,4,0.0,4






================== Typology17: all 163 variables


...

,question text,unique values,% missing,labelled codes
variable,,,,
respid,Individual unique respondent identification number,2002,0.0,0
sample,Sample type,2,0.0,2
int_date,"Interview date (NOTE: Final interviewer date, adjusted for interviews that are done early in the AM)",8,0.0,0
fcall,Date of first call (NOTE: different from SRBI's standard fcall variable),8,0.0,0
attempt,Number of call attempts,9,0.0,0
refusal,Refusal conversion,2,0.0,2
ilang,Language interview conducted in,2,0.0,2
cregion,Census region based on self-reported zipcode,4,0.0,4
state,State based on self-reported zipcode,51,0.0,58






================== W161: all 144 variables


,question text,unique values,% missing,labelled codes
variable,,,,
QKEY,QKEY_W161. Unique identifier,5086,0.0,0
INTERVIEW_START_W161,INTERVIEW_START_W161. Interview start time,4908,0.0,0
INTERVIEW_END_W161,INTERVIEW_END_W161. Interview end time,4891,0.0,0
DEVICE_TYPE_W161,DEVICE_TYPE_W161. The device used in the latest access of the survey link,3,0.0,4
SVYMODE_W161,SVYMODE_W161. Mode of survey,2,0.0,2
LANG_W161,LANG_W161. Language of interview,2,0.0,2
FORM_W161,"X_FORM_W161. Flag to randomly assign panelists to one of two forms (Form 1, Form 2) and weight within form ATP source: Randomly assigned for each survey.",2,0.0,2
X_PARTYSUMFINAL_W161,"X_PARTYSUMFINAL_W161. Note: Flag to identify republicans, republican leaners and non-leaners. Source: Frame file variable F_PARTYSUM_FINAL. Same codes as F_PARTYSUM_FINAL.",3,0.0,3
SATIS_W161,"SATIS_W161. All in all, are you satisfied or dissatisfied with the way things are going in this country today?",3,49.7,3






================== W167: all 170 variables


,question text,unique values,% missing,labelled codes
variable,,,,
QKEY,QKEY_W167. QKEY. Unique identifier,3589,0.0,0
INTERVIEW_START_W167,INTERVIEW_START_W167. Interview start time,3490,0.0,0
INTERVIEW_END_W167,INTERVIEW_END_W167. Interview end time,3522,0.0,0
DEVICE_TYPE_W167,DEVICE_TYPE_W167. The device used in the latest access of the survey link,3,0.0,4
SVYMODE_W167,SVYMODE_W167. Mode of survey,2,0.0,2
LANG_W167,LANG_W167. Language of interview,2,0.0,2
FORM_W167,"X_FORM_W167. Flag to randomly assign panelists to one of two forms (Form 1, Form 2) and weight within form ATP source: Randomly assigned for each survey.",2,0.0,2
POL1DT_W167,POL1DT_W167. Do you approve or disapprove of the way Donald Trump is handling his job as president?,3,0.0,3
POL1DTSTR_W167,POL1DTSTR_W167. Do you approve/disapprove of the way Donald Trump is handling his job as president...,3,0.8,3


## 4. Where Trump approval lives

### First: everything that mentions Trump

In [10]:
rows = []
for w in WAVE_ORDER:
    for c, l in labels[w].items():
        if l and "trump" in l.lower():
            rows.append({"wave": w, "variable": c, "question text": l})
trump_vars = pd.DataFrame(rows)
print(trump_vars.groupby("wave", sort=False).size().to_frame("variables that mention Trump").T)
trump_vars.set_index("wave")

wave                          Feb17  Apr17  Typology17  Jan18  Mar18  May18  \
variables that mention Trump     25     16           6     23      6     14   

wave                          Jun18  W161  W167  
variables that mention Trump     20    45    51  


,variable,question text
wave,,
Feb17,q1,Q.1 Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK. IF DEPENDS PROBE ONCE WITH: Overall do you approve or disapprove of the way Donald Trump is handling his job as President?
Feb17,q5af1,"Q.5aF1 Do you approve or disapprove of the way Donald Trump is handling [INSERT ITEM, RANDOMIZE] / The economy"
Feb17,q5bf1,"Q.5bF1 Do you approve or disapprove of the way Donald Trump is handling [INSERT ITEM, RANDOMIZE] / The threat of terrorism"
Feb17,q5cf1,"Q.5cF1 Do you approve or disapprove of the way Donald Trump is handling [INSERT ITEM, RANDOMIZE] / The nation's foreign policy"
Feb17,q5df1,"Q.5dF1 Do you approve or disapprove of the way Donald Trump is handling [INSERT ITEM, RANDOMIZE] / The nation's immigration policy"
Feb17,q6af2,Q.6aF2 Does Donald Trump make you feel [INSERT FIRST ITEM; RANDOMIZE] or not? / Hopeful
Feb17,q6bf2,Q.6bF2 Does Donald Trump make you feel [INSERT FIRST ITEM; RANDOMIZE] or not? / Proud
Feb17,q6cf2,Q.6cF2 Does Donald Trump make you feel [INSERT FIRST ITEM; RANDOMIZE] or not? / Angry
Feb17,q6df2,Q.6dF2 Does Donald Trump make you feel [INSERT FIRST ITEM; RANDOMIZE] or not? / Uneasy


There are many, but they are different questions: approval of Trump on specific issues (economy, tariffs, immigration...), opinions about his policies, about people who approve or disapprove of him, and so on.

In [11]:
def question_key(text):
    words = re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()
    while words and (words[0] in ("q", "qa") or any(ch.isdigit() for ch in words[0])):
        words.pop(0)
    return " ".join(words[:12])

trump_vars["key"] = trump_vars["question text"].map(question_key)
in_waves = trump_vars.groupby("key")["wave"].nunique().sort_values(ascending=False)
in_waves.head(6).to_frame("in how many files")

,in how many files
key,
do you approve or disapprove of the way donald trump is handling,9
how much respect do you think donald trump has for this country,3
as i read some pairs of opposite phrases tell me which one,2
do you think of your vote for congress as a vote for,2
do you think that donald trump has changed the republican party randomize,2
would you say you trust what donald trump says more about the,2


Only the **overall job approval** question ("Do you approve or disapprove of the way Donald Trump is handling his job as President?") is asked in all of them; the next most common Trump question appears in only a few files. The issue-specific questions are different in every survey. This is why we use overall job approval as the benchmark: it is the only Trump measure that can be followed from wave to wave.

### The approval question

In [12]:
rows = []
for w in WAVE_ORDER:
    for c, l in labels[w].items():
        if l and "trump" in l.lower() and "handling his job" in l.lower():
            rows.append({"wave": w, "variable": c, "question text": l[:110]})
approval_candidates = pd.DataFrame(rows)
approval_candidates

,wave,variable,question text
0,Feb17,q1,Q.1 Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK.
1,Apr17,q1,Q.1 Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK.
2,Typology17,qa1,Q.A1 Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK
3,Jan18,q2,Q.2. Do you approve or disapprove of the way Donald Trump is handling his job as President?
4,Mar18,q2,Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK. IF
5,May18,q2,Q.2. Do you approve or disapprove of the way Donald Trump is handling his job as President?
6,Jun18,q2,Q.2. Do you approve or disapprove of the way Donald Trump is handling his job as President?
7,W161,POL1DT_W161,POL1DT_W161. Do you approve or disapprove of the way Donald Trump is handling his job as president?
8,W161,POL1DTSTR_W161,POL1DTSTR_W161. Do you approve/disapprove of the way Donald Trump is handling his job as president...
9,W167,POL1DT_W167,POL1DT_W167. Do you approve or disapprove of the way Donald Trump is handling his job as president?


In some files the approval question is `q1`, in others it is `q2`

In [13]:
rows = []
for w in ["Feb17", "Apr17", "Jan18", "Mar18", "May18", "Jun18"]:
    rows.append({"wave": w, "q1": labels[w].get("q1", "")[:75], "q2": labels[w].get("q2", "")[:75]})
pd.DataFrame(rows).set_index("wave")

,q1,q2
wave,,
Feb17,Q.1 Do you approve or disapprove of the way Donald Trump is handling his jo,"Q.2 All in all, are you satisfied or dissatisfied with the way things are g"
Apr17,Q.1 Do you approve or disapprove of the way Donald Trump is handling his jo,"Q.2 All in all, are you satisfied or dissatisfied with the way things are g"
Jan18,"Q.1. All in all, are you satisfied or dissatisfied with the way things are",Q.2. Do you approve or disapprove of the way Donald Trump is handling his j
Mar18,"All in all, are you satisfied or dissatisfied with the way things are going",Do you approve or disapprove of the way Donald Trump is handling his job as
May18,"Q.1. All in all, are you satisfied or dissatisfied with the way things are",Q.2. Do you approve or disapprove of the way Donald Trump is handling his j
Jun18,"Q.1. All in all, are you satisfied or dissatisfied with the way things are",Q.2. Do you approve or disapprove of the way Donald Trump is handling his j


In Feb17 and Apr17 approval is `q1`, but in all 2018 files `q1` is "satisfied with the direction of the country" and approval is `q2`. If we merged by column name, we would mix two different questions.

## 5. How the answers are coded

In [14]:
APPROVAL_VAR = {}
for w in WAVE_ORDER:
    hits = approval_candidates[(approval_candidates["wave"] == w) &
                               approval_candidates["question text"].str.contains("approve or disapprove")]
    assert len(hits) == 1, (w, hits)
    APPROVAL_VAR[w] = hits["variable"].iloc[0]
APPROVAL_VAR

{'Feb17': 'q1',
 'Apr17': 'q1',
 'Typology17': 'qa1',
 'Jan18': 'q2',
 'Mar18': 'q2',
 'May18': 'q2',
 'Jun18': 'q2',
 'W161': 'POL1DT_W161',
 'W167': 'POL1DT_W167'}

In [15]:
tab = pd.DataFrame({w: surveys[w][APPROVAL_VAR[w]].value_counts(dropna=False) for w in WAVE_ORDER}).T
tab.columns = ["not asked (NaN)" if pd.isna(c) else int(c) for c in tab.columns]
tab = tab.fillna(0).astype(int)
print("Code labels in the files:")
print("  phone (Apr17):", vlabels("Apr17", APPROVAL_VAR["Apr17"]))
print("  ATP (W161):   ", vlabels("W161", APPROVAL_VAR["W161"]))
tab

Code labels in the files:
  phone (Apr17): {1.0: 'Approve', 2.0: 'Disapprove', 9.0: "Don't know/Refused (VOL.)"}
  ATP (W161):    {1.0: 'Approve', 2.0: 'Disapprove', 99.0: "Don't know/Refused/Web blank"}


,1,2,9,99,not asked (NaN)
Feb17,572,861,70,0,0
Apr17,606,819,76,0,0
Typology17,985,1385,134,0,2505
Jan18,563,854,86,0,0
Mar18,595,796,75,0,0
May18,614,804,85,0,0
Jun18,823,1072,107,0,0
W161,2273,2726,0,87,0
W167,1384,2178,0,27,0


In Typology17, 2,505 respondents have NaN

## 6. Date formats

In [16]:
rows = []
for w in WAVE_ORDER:
    for c, l in labels[w].items():
        if l and any(k in l.lower() for k in ["interview date", "interview start", "first call"]):
            rows.append({"wave": w, "variable": c, "label": l[:60], "dtype": str(surveys[w][c].dtype),
                         "first values": surveys[w][c].dropna().iloc[:2].tolist()})
date_candidates = pd.DataFrame(rows)
date_candidates

,wave,variable,label,dtype,first values
0,Feb17,int_date,Interview date,float64,"[21017.0, 21217.0]"
1,Feb17,fcall,Date of first call to number,float64,"[170207.0, 170207.0]"
2,Apr17,int_date,Interview date,float64,"[40517.0, 40517.0]"
3,Apr17,fcall,Date of first call to number,float64,"[170405.0, 170405.0]"
4,Typology17,int_date,Interview date,float64,"[61417.0, 60817.0]"
5,Jan18,int_date,"Interview date (NOTE: Final interviewer date, adjusted for i",float64,"[180110.0, 180110.0]"
6,Jan18,fcall,Date of first call (NOTE: different from SRBI's standard fca,float64,"[180110.0, 180110.0]"
7,Mar18,ftcalldt,Date of first call attempt,object,"[2018-03-08, 2018-03-12]"
8,May18,int_date,"Interview date (NOTE: Final interviewer date, adjusted for i",float64,"[180425.0, 180425.0]"
9,May18,fcall,Date of first call (NOTE: different from SRBI's standard fca,float64,"[180425.0, 180425.0]"


Most files have both an interview date (`int_date`) and the date of the first call (`fcall`). **March18 has only the date of the first call** (`ftcalldt`), and the ATP files have an interview start time as text.

In [17]:
DATE_VAR = {}
for w in WAVE_ORDER:
    cand = date_candidates[date_candidates["wave"] == w]
    interview = cand[cand["label"].str.contains("Interview date|Interview start", case=False)]
    DATE_VAR[w] = (interview if len(interview) else cand)["variable"].iloc[0]
DATE_VAR

{'Feb17': 'int_date',
 'Apr17': 'int_date',
 'Typology17': 'int_date',
 'Jan18': 'int_date',
 'Mar18': 'ftcalldt',
 'May18': 'int_date',
 'Jun18': 'int_date',
 'W161': 'INTERVIEW_START_W161',
 'W167': 'INTERVIEW_START_W167'}

The 2017 and 2018 dates are just numbers (`40517`, `180110`). We try two hypotheses on each of them: **MMDDYY** (with the leading zero missing, so `40517` -> `040517`) and **YYMMDD**, and see which one gives sensible dates for a file named e.g. `Apr17`:

In [18]:
def try_format(s, fmt):
    txt = s.dropna().astype("int64").astype(str).str.zfill(6) 
    return pd.to_datetime(txt, format=fmt, errors="coerce")

rows = []
for w in WAVE_ORDER:
    s = surveys[w][DATE_VAR[w]]
    if s.dtype == "float64":         
        for fmt, name in [("%m%d%y", "MMDDYY"), ("%y%m%d", "YYMMDD")]:
            d = try_format(s, fmt)
            rows.append({"wave": w, "raw example": int(s.iloc[0]), "format tried": name,
                         "not parsed": int(d.isna().sum()), "earliest": d.min(), "latest": d.max()})
pd.DataFrame(rows).set_index("wave")

,raw example,format tried,not parsed,earliest,latest
wave,,,,,
Feb17,21017,MMDDYY,0,2017-02-07,2017-02-12
Feb17,21017,YYMMDD,0,2002-07-17,2002-12-17
Apr17,40517,MMDDYY,0,2017-04-05,2017-04-11
Apr17,40517,YYMMDD,0,2004-05-17,2004-11-17
Typology17,61417,MMDDYY,0,2017-06-08,2017-07-09
Typology17,61417,YYMMDD,2210,2006-08-17,2007-09-17
Jan18,180110,MMDDYY,1503,NaT,NaT
Jan18,180110,YYMMDD,0,2018-01-10,2018-01-16
May18,180425,MMDDYY,1503,NaT,NaT


Only one hypothesis works for each file:
- 2017 files (Feb17, Apr17, Typology17): **MMDDYY**. `40517` is 05.04.2017. With YYMMDD we would get dates in 2002-2007, and for Typology17 more than half of the values would not parse at all.
- 2018 files (Jan18, May18, Jun18): **YYMMDD**. `180110` is 10.01.2018. With MMDDYY the month would be 18, which does not exist, so nothing parses.
- March18: the first-call date is already a proper date.
- ATP: a string like `'28-Jan-2025 21:38:38'`.

In [19]:
DATE_FMT = {"Feb17": "mddyy", "Apr17": "mddyy", "Typology17": "mddyy",
            "Jan18": "yymmdd", "Mar18": "date", "May18": "yymmdd", "Jun18": "yymmdd",
            "W161": "atp", "W167": "atp"}

def parse_dates(s, fmt):
    if fmt in ("mddyy", "yymmdd"):
        txt = s.dropna().astype("int64").astype(str).str.zfill(6)      # 40517 -> "040517"
        out = pd.to_datetime(txt, format="%m%d%y" if fmt == "mddyy" else "%y%m%d")
    elif fmt == "date":
        out = pd.to_datetime(s.dropna())                               # already a date
    elif fmt == "atp":
        out = pd.to_datetime(s.dropna(), format="%d-%b-%Y %H:%M:%S")   # '28-Jan-2025 21:38:38'
    return out.reindex(s.index).dt.normalize()                         # drop the time of day

rows = []
for w in WAVE_ORDER:
    parsed = parse_dates(surveys[w][DATE_VAR[w]], DATE_FMT[w])
    rows.append({"wave": w, "variable": DATE_VAR[w], "format": DATE_FMT[w],
                 "earliest": parsed.min().date(), "latest": parsed.max().date(),
                 "field days": (parsed.max() - parsed.min()).days})
pd.DataFrame(rows).set_index("wave")

,variable,format,earliest,latest,field days
wave,,,,,
Feb17,int_date,mddyy,2017-02-07,2017-02-12,5
Apr17,int_date,mddyy,2017-04-05,2017-04-11,6
Typology17,int_date,mddyy,2017-06-08,2017-07-09,31
Jan18,int_date,yymmdd,2018-01-10,2018-01-16,6
Mar18,ftcalldt,date,2018-03-07,2018-03-13,6
May18,int_date,yymmdd,2018-04-25,2018-05-01,6
Jun18,int_date,yymmdd,2018-06-05,2018-06-12,7
W161,INTERVIEW_START_W161,atp,2025-01-27,2025-02-02,6
W167,INTERVIEW_START_W167,atp,2025-04-07,2025-04-13,6


Every wave lasted 5-7 days, except Typology17: its field period is 31 days.

## 7. Typology-17: the approval question was not asked to everyone

The file has a phase variable (Phase A / Phase B).

In [20]:
ty = surveys["Typology17"].copy()
ty["date"] = parse_dates(ty["int_date"], "mddyy")
ty["asked approval question (qa1)"] = ty["qa1"].notna()
ty["phase"] = ty["phase"].map(vlabels("Typology17", "phase"))

print(pd.crosstab(ty["phase"], ty["asked approval question (qa1)"]))
print()
print(ty.groupby("phase")["date"].agg(["min", "max", "count"]))

asked approval question (qa1)  False  True 
phase                                      
Phase A                            0   2504
Phase B                         2505      0

               min        max  count
phase                               
Phase A 2017-06-08 2017-06-18   2504
Phase B 2017-06-27 2017-07-09   2505


The approval question was asked **only in Phase A** (June 8-18, 2017, 2,504 respondents). Phase B (June 27 - July 9) never received it. The topline PDF says the same: the question is labelled "ASK ALL PHASE A". So for approval we will keep only respondents with a non-empty `qa1`.

## 8. Weights

In [21]:
rows = []
for w in WAVE_ORDER:
    for c in surveys[w].columns:
        if "weight" in c.lower():
            rows.append({"wave": w, "variable": c, "label": (labels[w].get(c) or "")[:60]})
pd.DataFrame(rows).set_index("wave")

,variable,label
wave,,
Feb17,cellweight,Weight for cell sample
Feb17,weight,Final Weight
Apr17,utweight,Untruncated weight
Apr17,weight,Final Weight
Typology17,weight,Final combined weight
Jan18,cellweight,Final cellweight
Jan18,weight,Final weight
Mar18,cellweight,Final cell weight
Mar18,weight,Final weight


Some phone files have several weights, and we want the **final** one: it is called `weight` in the phone files and `WEIGHT_W161` / `WEIGHT_W167` in ATP.

In [22]:
WEIGHT_VAR = {w: ("weight" if w in phone else f"WEIGHT_{w}") for w in WAVE_ORDER}

pd.DataFrame({
    w: {"variable": WEIGHT_VAR[w], "label": labels[w][WEIGHT_VAR[w]][:25], "n": len(surveys[w]),
        "sum of weights": round(surveys[w][WEIGHT_VAR[w]].sum()),
        "min": round(surveys[w][WEIGHT_VAR[w]].min(), 2),
        "max": round(surveys[w][WEIGHT_VAR[w]].max(), 2)}
    for w in WAVE_ORDER
}).T

,variable,label,n,sum of weights,min,max
Feb17,weight,Final Weight,1503,4852,1.0,7.67
Apr17,weight,Final Weight,1501,4319,1.0,6.68
Typology17,weight,Final combined weight,5009,16254,1.0,8.45
Jan18,weight,Final weight,1503,1503,0.3,2.38
Mar18,weight,Final weight,1466,1466,0.23,2.94
May18,weight,Final weight,1503,1503,0.34,2.53
Jun18,weight,Final weight,2002,2002,0.31,2.43
W161,WEIGHT_W161,Wave 161 weight,5086,5086,0.16,3.5
W167,WEIGHT_W167,Wave 167 weight,3589,3589,0.13,2.72


The weight scales differ. In 2017 the sums are 4,852 / 4,319 / 16,254, while in 2018 and 2025 the sum of weights equals the number of respondents. So we will compute percentages **within each wave**, and if waves are ever pooled, weights must first be rescaled within each wave (for example divided by the wave's mean weight, so that the mean is 1).

## 9. The "how strongly" follow-up question

After the approval question there is a follow-up: "very strongly or not so strongly?". In the tables above it comes right after the main question (`q1` -> `q1a`, `q2` -> `q2a`, `POL1DT` -> `POL1DTSTR`)

In [23]:
STRENGTH_VAR = {}
for w in WAVE_ORDER:
    cols = list(surveys[w].columns)
    STRENGTH_VAR[w] = None
    for c in cols[cols.index(APPROVAL_VAR[w]) + 1:]:
        codes = " ".join(str(v).lower() for v in vlabels(w, c).values())
        if "very strongly" in codes and "not so strongly" in codes:
            STRENGTH_VAR[w] = c
            break

pd.DataFrame([{"wave": w, "approval": APPROVAL_VAR[w], "follow-up": STRENGTH_VAR[w],
               "question text": (labels[w].get(STRENGTH_VAR[w]) or "-")[:75]} for w in WAVE_ORDER]).set_index("wave")

,approval,follow-up,question text
wave,,,
Feb17,q1,q1a,"Q.1a Do you [approve/disapprove] very strongly, or not so strongly?"
Apr17,q1,q1a,"Q.1a Do you [approve/disapprove] very strongly, or not so strongly?"
Typology17,qa1,qa1a,"Q.A1a Do you [approve/disapprove] very strongly, or not so strongly?"
Jan18,q2,q2a,"Q.2a. Do you [approve/disapprove] very strongly, or not so strongly?"
Mar18,q2,None,-
May18,q2,q2a,"Q.2a. Do you [approve/disapprove] very strongly, or not so strongly?"
Jun18,q2,None,-
W161,POL1DT_W161,POL1DTSTR_W161,POL1DTSTR_W161. Do you approve/disapprove of the way Donald Trump is handli
W167,POL1DT_W167,POL1DTSTR_W167,POL1DTSTR_W167. Do you approve/disapprove of the way Donald Trump is handli


The follow-up question exists in 7 of the 9 files. **Mar18 and Jun18 do not have it**, so approval strength will be missing for those two waves.

In [43]:
for w in WAVE_ORDER:
    if STRENGTH_VAR[w]:
        print(w, STRENGTH_VAR[w], vlabels(w, STRENGTH_VAR[w]))

Feb17 q1a {1.0: 'Very strongly', 2.0: 'Not so strongly', 9.0: "Don't know/Refused (VOL.)"}
Apr17 q1a {1.0: 'Very strongly', 2.0: 'Not so strongly', 9.0: "Don't know/Refused (VOL.)"}
Typology17 qa1a {1.0: 'Very strongly', 2.0: 'Not so strongly', 9.0: "Don't know/Refused (VOL.)"}
Jan18 q2a {1.0: 'Very strongly', 2.0: 'Not so strongly', 9.0: "(VOL) Don't know/Refused"}
May18 q2a {1.0: 'Very strongly', 2.0: 'Not so strongly', 9.0: "(VOL) Don't know/Refused"}
W161 POL1DTSTR_W161 {1.0: 'Very strongly', 2.0: 'Not so strongly', 99.0: "Don't know/Refused/Web blank"}
W167 POL1DTSTR_W167 {1.0: 'Very strongly', 2.0: 'Not so strongly', 99.0: "Don't know/Refused/Web blank"}


## 10. Party, ideology, education, age, gender

In [24]:
def find_vars(pattern):
    return pd.DataFrame({w: [", ".join(c for c in surveys[w].columns if re.match(pattern, c, re.I))]
                         for w in WAVE_ORDER}, index=["variables found"]).T

def describe(wave, pattern, cut=22):
    rows = [{"variable": c, "question text": (labels[wave].get(c) or "")[:70],
             "codes": {k: s[:cut] for k, s in vlabels(wave, c).items()}}
            for c in surveys[wave].columns if re.match(pattern, c, re.I)]
    return pd.DataFrame(rows).set_index("variable")

def show_choice(varmap, cut=28):
    return pd.DataFrame([{"wave": w, "variable": v, "question text": (labels[w].get(v) or "")[:55],
                          "codes": {k: s[:cut] for k, s in vlabels(w, v).items()}}
                         for w, v in varmap.items()]).set_index("wave")

In [25]:
rows = []
for w in WAVE_ORDER:
    for c, l in labels[w].items():
        if l and "unique" in l.lower():
            rows.append({"wave": w, "variable": c, "question text": l[:70], "all values different": surveys[w][c].is_unique})
id_candidates = pd.DataFrame(rows).set_index("wave")
ID_VAR = id_candidates["variable"].to_dict()
id_candidates

,variable,question text,all values different
wave,,,
Feb17,psraid,Unique ID,True
Apr17,psraid,Unique ID,True
Typology17,mergeid,Unique ID after merge,True
Jan18,respid,Individual unique respondent identification number,True
Mar18,masterid,Imported from sample file: Unique Respondent ID,True
May18,respid,Individual unique respondent identification number,True
Jun18,respid,Individual unique respondent identification number,True
W161,QKEY,QKEY_W161. Unique identifier,True
W167,QKEY,QKEY_W167. QKEY. Unique identifier,True


One ID variable per file, and all its values are different, so it identifies a respondent within a wave. The names are again different (`psraid`, `respid`, `masterid`, `mergeid`, `QKEY`).

In [26]:
find_vars(r"(f_)?party")

,variables found
Feb17,"party, partyln, partysum, partyideo"
Apr17,"party, partyln, partysum, partyideo"
Typology17,"party, partyln, partystr, partysum, partyideo"
Jan18,"party, partyln, partysum, partyideo"
Mar18,"party, partyln, partysum, partyideo"
May18,"party, partyln, partysum, partyideo, partysumideo"
Jun18,"party, partyln, partysum, partyideo, partysumideo"
W161,"F_PARTY_FINAL, F_PARTYLN_FINAL, F_PARTYSTR_FINAL, F_PARTYLNCLOSE_FINAL, F_PARTYSUM_FINAL, F_PARTYSUMIDEO_FINAL"
W167,"F_PARTY_FINAL, F_PARTYLN_FINAL, F_PARTYSTR_FINAL, F_PARTYLNCLOSE_FINAL, F_PARTYSUM_FINAL, F_PARTYSUMIDEO_FINAL"


In [27]:
print("Apr17 (phone):")
display(describe("Apr17", r"(f_)?party"))
print("W161 (ATP):")
display(describe("W161", r"(f_)?party"))

Apr17 (phone):


,question text,codes
variable,,
party,"PARTY. In politics TODAY, do you consider yourself a Republican, Democ","{1.0: 'Republican', 2.0: 'Democrat', 3.0: 'Independent', 4.0: 'No preference (VOL.)', 5.0: 'Other party (VOL.)', 9.0: 'Don't know/Refused (VO'}"
partyln,PARTYLN. As of today do you lean more to the Republican Party or more,"{1.0: 'Republican', 2.0: 'Democrat', 9.0: 'Other/Don't know/Refus'}"
partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
partyideo,Party and Ideology,"{1.0: 'Cons Rep', 2.0: 'Mod-Lib Rep', 3.0: 'Ind', 4.0: 'Cons-Mod Dem', 5.0: 'Lib Dem', 9.0: 'DK/Ref to party or ide'}"


W161 (ATP):


,question text,codes
variable,,
F_PARTY_FINAL,Party,"{1.0: 'Republican', 2.0: 'Democrat', 3.0: 'Independent', 4.0: 'Something else', 99.0: 'Refused'}"
F_PARTYLN_FINAL,Party lean,"{1.0: 'The Republican Party', 2.0: 'The Democratic Party', 99.0: 'Refused'}"
F_PARTYSTR_FINAL,Party strength,"{1.0: 'Strongly', 2.0: 'Not strongly', 99.0: 'Refused'}"
F_PARTYLNCLOSE_FINAL,Party closeness among leaners,"{1.0: 'Extremely close', 2.0: 'Very close', 3.0: 'Somewhat close', 4.0: 'Not too close', 5.0: 'Not at all close', 99.0: 'Refused'}"
F_PARTYSUM_FINAL,Party summary,"{1.0: 'Rep/Lean Rep', 2.0: 'Dem/Lean Dem', 9.0: 'DK/Refused/No lean'}"
F_PARTYSUMIDEO_FINAL,Combining ideology and party identification,"{1.0: 'Conservative Rep/Lean', 2.0: 'Moderate/Liberal Rep/L', 3.0: 'Moderate/Conservative ', 4.0: 'Liberal Dem/Lean', 9.0: 'Refused either F_IDEO '}"


So `party` is the plain party question, `partyln` is the follow-up "which party do you lean to", `partystr` is strength, `partyideo` mixes party with ideology, and `partysum` (`F_PARTYSUM_FINAL` in ATP) is the **summary that puts leaners together with Republicans/Democrats**. That is what we need to split respondents into Rep/Dem. Let's check its text and codes in every file:

In [28]:
PARTY_VAR = {w: ("F_PARTYSUM_FINAL" if w in atp else "partysum") for w in WAVE_ORDER}
show_choice(PARTY_VAR)

,variable,question text,codes
wave,,,
Feb17,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
Apr17,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
Typology17,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
Jan18,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
Mar18,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
May18,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
Jun18,partysum,Leaned party identification,"{1.0: 'Rep/lean Rep', 2.0: 'Dem/lean Dem', 9.0: 'DK/Ref-no lean'}"
W161,F_PARTYSUM_FINAL,Party summary,"{1.0: 'Rep/Lean Rep', 2.0: 'Dem/Lean Dem', 9.0: 'DK/Refused/No lean'}"
W167,F_PARTYSUM_FINAL,Party summary,"{1.0: 'Rep/Lean Rep', 2.0: 'Dem/Lean Dem', 9.0: 'DK/Refused/No lean'}"


The codes are the same everywhere: 1 = Rep/lean Rep, 2 = Dem/lean Dem, 9 = other/DK. Only the variable names differ.

**Ideology.**

In [29]:
find_vars(r"(f_)?ideo")

,variables found
Feb17,ideo
Apr17,ideo
Typology17,"ideoconsist, ideo"
Jan18,ideo
Mar18,ideo
May18,ideo
Jun18,ideo
W161,F_IDEO
W167,F_IDEO


Let's see what these are (Apr17, Typology17 and W161 as examples):

In [30]:
print("Apr17 (phone):")
display(describe("Apr17", r".*ideo"))
print("Typology17:")
display(describe("Typology17", r"ideo"))
print("W161 (ATP):")
display(describe("W161", r".*ideo"))

Apr17 (phone):


,question text,codes
variable,,
ideo,"IDEO. In general, would you describe your political views as... [READ]","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 9.0: 'Don't know/Refused (VO'}"
partyideo,Party and Ideology,"{1.0: 'Cons Rep', 2.0: 'Mod-Lib Rep', 3.0: 'Ind', 4.0: 'Cons-Mod Dem', 5.0: 'Lib Dem', 9.0: 'DK/Ref to party or ide'}"


Typology17:


,question text,codes
variable,,
ideoconsist,Ideoconsist variable - Combining consCount10 and libCount10 - Phase A,{}
ideo,"IDEO. In general, would you describe your political views as... [READ]","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 9.0: 'Don't know/Refused (VO'}"


W161 (ATP):


,question text,codes
variable,,
F_PARTYSUMIDEO_FINAL,Combining ideology and party identification,"{1.0: 'Conservative Rep/Lean', 2.0: 'Moderate/Liberal Rep/L', 3.0: 'Moderate/Conservative ', 4.0: 'Liberal Dem/Lean', 9.0: 'Refused either F_IDEO '}"
F_IDEO,Ideology,"{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 99.0: 'Refused'}"


`ideo` (phone) and `F_IDEO` (ATP) are the plain ideology question. The others either combine ideology with party (`partyideo`, `partysumideo`, `F_PARTYSUMIDEO_FINAL`) or are an index made for the Typology study (`ideoconsist`), so we do not use them.

In [31]:
IDEO_VAR = {w: ("F_IDEO" if w in atp else "ideo") for w in WAVE_ORDER}
show_choice(IDEO_VAR)

,variable,question text,codes
wave,,,
Feb17,ideo,"IDEO. In general, would you describe your political vie","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 9.0: 'Don't know/Refused (VOL.)'}"
Apr17,ideo,"IDEO. In general, would you describe your political vie","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 9.0: 'Don't know/Refused (VOL.)'}"
Typology17,ideo,"IDEO. In general, would you describe your political vie","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 9.0: 'Don't know/Refused (VOL.)'}"
Jan18,ideo,"IDEO. In general, would you describe your political vie","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal [OR]', 5.0: 'Very liberal', 9.0: '(VOL) Don't know/Refused'}"
Mar18,ideo,"In general, would you describe your political views as.","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal [OR]', 5.0: 'Very liberal', 9.0: '[VOL. DO NOT READ] Don''t kn'}"
May18,ideo,"IDEO. In general, would you describe your political vie","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal [OR]', 5.0: 'Very liberal', 9.0: '(VOL) Don't know/Refused'}"
Jun18,ideo,"IDEO. In general, would you describe your political vie","{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal [OR]', 5.0: 'Very liberal', 9.0: '(VOL) Don't know/Refused'}"
W161,F_IDEO,Ideology,"{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 99.0: 'Refused'}"
W167,F_IDEO,Ideology,"{1.0: 'Very conservative', 2.0: 'Conservative', 3.0: 'Moderate', 4.0: 'Liberal', 5.0: 'Very liberal', 99.0: 'Refused'}"


Codes 1-5 mean the same in all files (very conservative ... very liberal). The DK code differs again: 9 in phone files, 99 in ATP.

**Education.**

In [32]:
find_vars(r"(f_)?educ")

,variables found
Feb17,educ2
Apr17,educ2
Typology17,educ2
Jan18,educ
Mar18,educ
May18,educ
Jun18,educ
W161,"F_EDUCCAT, F_EDUCCAT2"
W167,"F_EDUCCAT, F_EDUCCAT2"


In [33]:
EDUC_VAR = {w: ("F_EDUCCAT" if w in atp else ("educ" if "educ" in surveys[w].columns else "educ2")) for w in WAVE_ORDER}
show_choice(EDUC_VAR, cut=20)

,variable,question text,codes
wave,,,
Feb17,educ2,EDUC2. What is the highest level of school you have com,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: 'Don't know/Refused ('}"
Apr17,educ2,EDUC2. What is the highest level of school you have com,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: 'Don't know/Refused ('}"
Typology17,educ2,EDUC2. What is the highest level of school you have com,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: 'Don't know/Refused'}"
Jan18,educ,EDUC. What is the highest level of school you have comp,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: '(VOL) Don't know/Ref'}"
Mar18,educ,What is the highest level of school you have completed,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: 'Don''t know/Refused '}"
May18,educ,EDUC. What is the highest level of school you have comp,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: '(VOL) Don't know/Ref'}"
Jun18,educ,EDUC. What is the highest level of school you have comp,"{1.0: 'Less than high schoo', 2.0: 'High school incomple', 3.0: 'High school graduate', 4.0: 'Some college, no deg', 5.0: 'Two year associate d', 6.0: 'Four year college or', 7.0: 'Some postgraduate or', 8.0: 'Postgraduate or prof', 9.0: '(VOL) Don't know/Ref'}"
W161,F_EDUCCAT,Education level category,"{1.0: 'College graduate+', 2.0: 'Some College', 3.0: 'H.S. graduate or les', 99.0: 'Refused'}"
W167,F_EDUCCAT,Education level category,"{1.0: 'College graduate+', 2.0: 'Some College', 3.0: 'H.S. graduate or les', 99.0: 'Refused'}"


Phone files have 8 education levels (in the variable `educ`, or `educ2` in the 2017 files), ATP has only 3 groups (`F_EDUCCAT`). To compare them we will have to collapse the 8 levels into the same 3 groups.

**Age and gender.**

In [34]:
print("Age variables:")
display(find_vars(r"(f_)?age"))
print("Gender variables:")
display(find_vars(r"(sex|f_gender)"))
print("Phone age (Apr17), exact number:", surveys["Apr17"]["age"].describe().round(1).to_dict())
print("ATP age (W161), categories only:", vlabels("W161", "F_AGECAT"))
print("Phone gender:", vlabels("Apr17", "sex"))
print("ATP gender:  ", vlabels("W161", "F_GENDER"))

Age variables:


,variables found
Feb17,age
Apr17,age
Typology17,age
Jan18,age
Mar18,age
May18,age
Jun18,age
W161,F_AGECAT
W167,F_AGECAT


Gender variables:


,variables found
Feb17,sex
Apr17,sex
Typology17,sex
Jan18,sex
Mar18,sex
May18,sex
Jun18,sex
W161,F_GENDER
W167,F_GENDER


Phone age (Apr17), exact number: {'count': 1501.0, 'mean': 50.5, 'std': 18.8, 'min': 18.0, '25%': 34.0, '50%': 52.0, '75%': 65.0, 'max': 99.0}
ATP age (W161), categories only: {1.0: '18-29', 2.0: '30-49', 3.0: '50-64', 4.0: '65+', 99.0: 'Refused'}
Phone gender: {1.0: 'Male', 2.0: 'Female'}
ATP gender:   {1.0: 'A man', 2.0: 'A woman', 3.0: 'In some other way', 99.0: 'Refused'}


Phone files have exact age, ATP only age categories (18-29, 30-49, 50-64, 65+), so exact age will have to be converted into the same groups. Gender has one extra category in ATP ("in some other way").

In [35]:
AGE_VAR = {w: ("F_AGECAT" if w in atp else "age") for w in WAVE_ORDER}
GENDER_VAR = {w: ("F_GENDER" if w in atp else "sex") for w in WAVE_ORDER}

## 11. Survey mode

How were people interviewed? Phone files have a sample type variable (`sample`, or `samptype` in March18), ATP has `SVYMODE`:

In [36]:
MODE_VAR = {}
rows = []
for w in WAVE_ORDER:
    var = [c for c in ["sample", "samptype", f"SVYMODE_{w}"] if c in surveys[w].columns][0]
    MODE_VAR[w] = var
    counts = surveys[w][var].map(vlabels(w, var)).value_counts().to_dict()
    rows.append({"wave": w, "variable": var, "n": len(surveys[w]), "breakdown": counts})
pd.DataFrame(rows).set_index("wave")

,variable,n,breakdown
wave,,,
Feb17,sample,1503,"{'Cell': 1126, 'Landline': 377}"
Apr17,sample,1501,"{'Cell': 1126, 'Landline': 375}"
Typology17,sample,5009,"{'Cell': 3754, 'Landline': 1255}"
Jan18,sample,1503,"{'Cell phone': 1127, 'Landline': 376}"
Mar18,samptype,1466,"{'Cell Phone': 1082, 'Landline': 384}"
May18,sample,1503,"{'Cell phone': 1127, 'Landline': 376}"
Jun18,sample,2002,"{'Cell phone': 1500, 'Landline': 502}"
W161,SVYMODE_W161,5086,"{'Web': 4893, 'CATI': 193}"
W167,SVYMODE_W167,3589,"{'Web': 3465, 'CATI': 124}"


For the 2017–2018 surveys, all respondents were interviewed by telephone, although the raw files distinguish landline and cell-phone samples. Since we only need a broad collection-method indicator, the combined dataset stores these waves as phone_rdd. For ATP waves, where respondents may answer either online or by phone, we retain SVYMODE.

## 12. A draft crosswalk and what the merging code will need to do

Everything we chose above can be collected into one table: for every wave, which raw variable plays which role

In [37]:
crosswalk = pd.DataFrame({
    "id": ID_VAR, "approval": APPROVAL_VAR, "strength": STRENGTH_VAR, "weight": WEIGHT_VAR,
    "date": DATE_VAR, "date format": DATE_FMT, "party": PARTY_VAR, "ideology": IDEO_VAR,
    "education": EDUC_VAR, "age": AGE_VAR, "gender": GENDER_VAR, "mode": MODE_VAR,
})
crosswalk.index.name = "wave"
crosswalk

,id,approval,strength,weight,date,date format,party,ideology,education,age,gender,mode
wave,,,,,,,,,,,,
Feb17,psraid,q1,q1a,weight,int_date,mddyy,partysum,ideo,educ2,age,sex,sample
Apr17,psraid,q1,q1a,weight,int_date,mddyy,partysum,ideo,educ2,age,sex,sample
Typology17,mergeid,qa1,qa1a,weight,int_date,mddyy,partysum,ideo,educ2,age,sex,sample
Jan18,respid,q2,q2a,weight,int_date,yymmdd,partysum,ideo,educ,age,sex,sample
Mar18,masterid,q2,None,weight,ftcalldt,date,partysum,ideo,educ,age,sex,samptype
May18,respid,q2,q2a,weight,int_date,yymmdd,partysum,ideo,educ,age,sex,sample
Jun18,respid,q2,None,weight,int_date,yymmdd,partysum,ideo,educ,age,sex,sample
W161,QKEY,POL1DT_W161,POL1DTSTR_W161,WEIGHT_W161,INTERVIEW_START_W161,atp,F_PARTYSUM_FINAL,F_IDEO,F_EDUCCAT,F_AGECAT,F_GENDER,SVYMODE_W161
W167,QKEY,POL1DT_W167,POL1DTSTR_W167,WEIGHT_W167,INTERVIEW_START_W167,atp,F_PARTYSUM_FINAL,F_IDEO,F_EDUCCAT,F_AGECAT,F_GENDER,SVYMODE_W167


The same 4 columns from every file, side by side (first 3 rows), show why the crosswalk is needed. Same content, different names and codes:

In [38]:
for w in WAVE_ORDER:
    print("===", w)
    display(surveys[w][[APPROVAL_VAR[w], WEIGHT_VAR[w], DATE_VAR[w], PARTY_VAR[w]]].head(3))

=== Feb17


,q1,weight,int_date,partysum
0,2.0,1.733333,21017.0,2.0
1,2.0,1.500000,21217.0,2.0
2,2.0,1.533333,21217.0,2.0


=== Apr17


,q1,weight,int_date,partysum
0,2.0,2.941176,40517.0,2.0
1,1.0,1.323529,40517.0,1.0
2,2.0,1.235294,40517.0,2.0


=== Typology17


...

,q2,weight,int_date,partysum
0,2.0,0.581332,180605.0,2.0
1,9.0,1.559586,180605.0,1.0
2,2.0,2.428216,180605.0,1.0


=== W161


,POL1DT_W161,WEIGHT_W161,INTERVIEW_START_W161,F_PARTYSUM_FINAL
0,1.0,1.046094,28-Jan-2025 21:38:38,1.0
1,1.0,0.992757,29-Jan-2025 12:54:51,1.0
2,2.0,1.196266,28-Jan-2025 16:24:57,2.0


=== W167


,POL1DT_W167,WEIGHT_W167,INTERVIEW_START_W167,F_PARTYSUM_FINAL
0,2.0,1.135669,08-Apr-2025 15:12:07,2.0
1,1.0,1.169792,08-Apr-2025 14:50:19,1.0
2,1.0,1.110668,08-Apr-2025 17:06:04,1.0


### Why only these columns?

Every file has 108-178 columns, and we keep 10 or 11 of them (10 in Mar18 and Jun18, which have no "how strongly" follow-up).

In [39]:
roles = {"id": ID_VAR, "approval": APPROVAL_VAR, "strength": STRENGTH_VAR, "weight": WEIGHT_VAR, "date": DATE_VAR,
         "party": PARTY_VAR, "ideology": IDEO_VAR, "education": EDUC_VAR, "age": AGE_VAR, "gender": GENDER_VAR,
         "mode": MODE_VAR}
role_of = {(w, var): role for role, m in roles.items() for w, var in m.items() if var}
codebook["role"] = [role_of.get((w, v), "") for w, v in zip(codebook["wave"], codebook["variable"])]

kept_summary = codebook.groupby("wave", sort=False).agg(total=("variable", "size"), kept=("role", lambda r: int((r != "").sum())))
kept_summary["dropped"] = kept_summary["total"] - kept_summary["kept"]
kept_summary["% kept"] = (100 * kept_summary["kept"] / kept_summary["total"]).round(1)
kept_summary.T

wave,Feb17,Apr17,Typology17,Jan18,Mar18,May18,Jun18,W161,W167
total,127.0,178.0,163.0,168.0,108.0,123.0,130.0,144.0,170.0
kept,11.0,11.0,11.0,11.0,10.0,11.0,10.0,11.0,11.0
dropped,116.0,167.0,152.0,157.0,98.0,112.0,120.0,133.0,159.0
% kept,8.7,6.2,6.7,6.5,9.3,8.9,7.7,7.6,6.5


The kept variables, with their question text (this is the full list of what goes into the merged dataset):

In [40]:
codebook[codebook["role"] != ""][["wave", "role", "variable", "question text"]].set_index("wave")

,role,variable,question text
wave,,,
Feb17,id,psraid,Unique ID
Feb17,mode,sample,Sample type
Feb17,date,int_date,Interview date
Feb17,approval,q1,Q.1 Do you approve or disapprove of the way Donald Trump is handling his job as President? [IF DK ENTER AS DK. IF DEPENDS PROBE ONCE WITH: Overall do you approve or disapprove of the way Donald Trump is handling his job as President?
Feb17,strength,q1a,"Q.1a Do you [approve/disapprove] very strongly, or not so strongly?"
Feb17,gender,sex,SEX. Respondent's sex [DO NOT ASK]
Feb17,age,age,AGE. What is your age?
Feb17,education,educ2,EDUC2. What is the highest level of school you have completed or the highest degree you have received? [DO NOT READ] [INTERVIEWER NOTE: Enter code 3-HS grad if R completed training that did NOT count toward a degree]
Feb17,party,partysum,Leaned party identification


**Why not all the other columns?**

In [41]:
codebook["name"] = codebook["variable"].str.lower()
waves_per_name = codebook.groupby("name")["wave"].nunique()

print("Different column names in all files together:", len(waves_per_name))
print("Names that exist in only ONE file:", int((waves_per_name == 1).sum()))
print("Names that exist in 2 or more files:", int((waves_per_name >= 2).sum()))
print("Names that exist in 5 or more files:", int((waves_per_name >= 5).sum()))

Different column names in all files together: 888
Names that exist in only ONE file: 716
Names that exist in 2 or more files: 172
Names that exist in 5 or more files: 46


 888
Names that exist in only ONE file: 716
Names that exist in 2 or more files: 172
Names that exist in 5 or more files: 46


Most columns exist in only one file. They are questions on topics that were asked in one survey only (tax law, tariffs, specific news events...), so they cannot be compared across waves. 

The columns that do repeat in many files are these:

In [42]:
shared = waves_per_name[waves_per_name >= 5].sort_values(ascending=False)
first_text = codebook.drop_duplicates("name").set_index("name")["question text"]
pd.DataFrame({"in how many files": shared, "question text (in the first file that has it)": first_text[shared.index].str[:90]})

,in how many files,question text (in the first file that has it)
name,,
age,7,AGE. What is your age?
attend,7,"ATTEND. Aside from weddings and funerals, how often do you attend religious services... mo"
born,7,"BORN. Would you describe yourself as a 'born again' or evangelical Christian, or not?"
birth_hisp,7,"BIRTH_HISP. Were you born in the United States, on the island of Puerto Rico, or in anothe"
cp,7,Resp has a cell phone
chr,7,CHR. Do you think of yourself as a Christian or not? [IF R NAMED A NON-CHRISTIAN RELIGION
hh3,7,"HH3. How many, including yourself, are adults, age 18 and older?"
hh1,7,"HH1. How many people, including yourself, live in your household?"
cregion,7,Final Census Region after zip/fips merges


The columns that repeat are mostly **technical or screening**  or **basic demographics** 

So we keep only what we needs:
1. **Trump job approval** and its "how strongly" follow-up: the measure we want to follow over time.
2. **Weight, date and ID**: to compute weighted shares per wave and know when the interview took place.
3. **Party, ideology, education, age, gender**: to compare groups of people
4. **Survey mode**: because phone and online panel results are not fully comparable.

Everything else is either a one-off question about a specific topic or a fieldwork detail. If later we need more, we add them.